# RSNA Knee Abnormality Detection — INFERENCE ONLY
### DINOv2-336 (LoRA fine-tuned) + Model G, 5-fold probability-mean ensemble

This notebook does **not** train, validate, cross-validate, ablate, or produce any new
checkpoints. It only:

1. Downloads the already-trained DINOv2-336 LoRA backbone and the 5 Model G fold
   checkpoints from Hugging Face (`micheal22/l_i_m_l`).
2. Reproduces — **unchanged** — the exact preprocessing and architecture from the
   original training notebook (`notebook3f80cddf2d.ipynb`), which is the source of
   truth for every detail below.
3. Runs MRI-only inference (no radiology reports, no LLM labels) on the Kaggle test set.
4. Writes `/kaggle/working/submission.csv`.

Primary experiment reproduced: **DINOv2-336, LoRA fine-tuned, baseline intensity
normalization, laterality canonicalization = TRUE, Model G, 5-fold mean ensemble**
(reference gold OOF macro AUC ≈ 0.7966).


## 1. Setup / imports

In [1]:
import os, sys, json, math, time, glob, warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

N_GPUS = torch.cuda.device_count()
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"torch.cuda.device_count() = {N_GPUS}")
print(f"Using device: {DEVICE}")
for i in range(N_GPUS):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")


torch.cuda.device_count() = 2
Using device: cuda:0
  GPU 0: Tesla T4
  GPU 1: Tesla T4


## 2. Configuration
Values below are copied **unchanged** from the training notebook's configuration cell —
`LABELS`, `MODEL_G_CONFIG`, `METADATA_DIM`, the LoRA hyperparameters, and the
preprocessing flags all have to match exactly or the checkpoints will not load /
will not reproduce the training-time representation.

In [2]:
# ---------------- Competition / test data paths ----------------
# NOTE: the original notebook only ever loaded TRAIN paths (test data is not
# available at training time on Kaggle). These TEST_* paths follow the same
# competition directory convention the training notebook used for train data;
# verify them against the actual competition data tab before `Run All`.
COMPETITION_DIR   = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
TEST_CSV          = f"{COMPETITION_DIR}/test.csv"
TEST_SERIES_CSV   = f"{COMPETITION_DIR}/test_series.csv"
SAMPLE_SUBMISSION = f"{COMPETITION_DIR}/sample_submission.csv"

# Raw test DICOMs -- volumes are built directly from these (NOT the pre-baked
# "processed 3D volume" NPZ dataset; this notebook now reads the actual .dcm files).
TEST_DICOM_DIR  = f"{COMPETITION_DIR}/test_series"

DINOV2_LOCAL_PATH = "/kaggle/input/models/metaresearch/dinov2/pytorch/base/1"

WORK_DIR = Path("/kaggle/working")
INFERENCE_CACHE_ROOT = WORK_DIR / "inference_cache"          # NEVER the training cache dir
CACHE_DINO336_FINETUNE = INFERENCE_CACHE_ROOT / "dino336_finetuned"
CKPT_DIR = WORK_DIR / "downloaded_checkpoints"
for d in [INFERENCE_CACHE_ROOT, CACHE_DINO336_FINETUNE, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---------------- Labels (exact 12, multi-label, sigmoid, never softmax) ----------------
LABELS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]
N_LABELS = len(LABELS)

# ---------------- Exact Model G architecture config (verbatim from training) ----------------
MODEL_G_CONFIG = {
    "embed_dim": 768,
    "hidden_dim": 384,
    "view_dim": 256,
    "dropout": 0.20,
    "planes": ["Sagittal", "Coronal", "Axial"],
    "use_view_projection": True,
    "use_label_specific_gates": True,
    "classifier_hidden_dim": 128,
}
METADATA_DIM = 3  # [fluid_sensitive, fat_suppression, slice_fraction] -- exact, do not change

N_FOLDS = 5

# ---------------- Primary-experiment preprocessing flags (fixed, not swept here) ----------------
USE_LATERALITY_CANONICALIZATION = True
INTENSITY_NORMALIZATION = "baseline"
REPRESENTATION = "cls"          # Model G was trained on the CLS representation
MAX_SLICES_FOR_METADATA_NORM = 60
MAX_SERIES_PER_STUDY = 8
MAX_SLICES_PER_SERIES = 48

# ---------------- Exact LoRA config used to fine-tune the backbone ----------------
LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGET_KEYWORDS = ["query", "key", "value", "dense"]
LORA_TUNE_LAYERNORMS = True

DINO_RESOLUTION = 336
DINO_BATCH_SIZE = 32   # lower if GPU memory is tight

# Each raw DICOM slice is resized to this (H, W) when building the in-memory volume.
# Matches DINO_RESOLUTION (336) directly -- the DINOv2-336 model is what actually
# consumes these slices, so there's no separate 224-sized intermediate anymore.
VOLUME_RESIZE_HW = DINO_RESOLUTION

# ---------------- Hugging Face source of the trained weights ----------------
HF_REPO_ID = "micheal22/l_i_m_l"
HF_BACKBONE_FILE = "backbone/dinov2_336_lora_finetuned_backbone.pt"
HF_FOLD_FILES = [f"model_g_checkpoints/dino336_finetuned_fold_{i}_best.pt" for i in range(N_FOLDS)]

SMOKE_TEST = True
SMOKE_N_STUDIES = 2

print("Configuration loaded. Primary experiment: DINOv2-336 LoRA + Model G, 5-fold mean ensemble.")


Configuration loaded. Primary experiment: DINOv2-336 LoRA + Model G, 5-fold mean ensemble.


## 3. Load test metadata

In [3]:
def safe_read_csv(path, **kwargs):
    if not os.path.exists(path):
        print(f"WARNING: {path} not found in this environment.")
        return None
    return pd.read_csv(path, **kwargs)

test_df        = safe_read_csv(TEST_CSV)
test_series_df = safe_read_csv(TEST_SERIES_CSV)
sample_sub_df  = safe_read_csv(SAMPLE_SUBMISSION)

for name, df in [("test", test_df), ("test_series", test_series_df), ("sample_submission", sample_sub_df)]:
    if df is not None:
        print(f"{name}: shape={df.shape}, columns={list(df.columns)}")
    else:
        print(f"{name}: NOT AVAILABLE")

assert test_series_df is not None, "test_series.csv is required to build the SeriesInstanceUID -> StudyInstanceUID mapping."
assert {"SeriesInstanceUID", "StudyInstanceUID"}.issubset(test_series_df.columns), \
    "test_series.csv must contain SeriesInstanceUID and StudyInstanceUID"

test_series_df["StudyInstanceUID"]  = test_series_df["StudyInstanceUID"].astype(str).str.strip()
test_series_df["SeriesInstanceUID"] = test_series_df["SeriesInstanceUID"].astype(str).str.strip()

series_to_study = dict(zip(test_series_df["SeriesInstanceUID"], test_series_df["StudyInstanceUID"]))
study_to_series = {}
for s, st in series_to_study.items():
    study_to_series.setdefault(st, []).append(s)

# The list of studies we must predict for comes from sample_submission.csv -- the
# authoritative "what needs a row" source -- NOT from test_series.csv. The two are
# expected to line up, but on a hidden test set that's "larger/smaller/different"
# they are not guaranteed to match 1:1 (e.g. a study listed in the submission with
# no rows in test_series.csv). Deriving test_study_uids from test_series.csv alone
# silently drops such studies, which later blows up the final submission asserts.
if sample_sub_df is not None and "StudyInstanceUID" in sample_sub_df.columns:
    test_study_uids = sorted(sample_sub_df["StudyInstanceUID"].astype(str).str.strip().unique())
else:
    print("WARNING: sample_submission.csv unavailable/missing StudyInstanceUID -- "
          "falling back to studies derived from test_series.csv (coverage may be incomplete).")
    test_study_uids = sorted(study_to_series.keys())

print(f"Test studies to predict (from sample_submission.csv when available): {len(test_study_uids)}")
print(f"Test series:  {len(series_to_study)}")

missing_series_metadata = sorted(set(test_study_uids) - set(study_to_series.keys()))
if missing_series_metadata:
    print(f"WARNING: {len(missing_series_metadata)} studies to predict have NO series metadata "
          f"in test_series.csv at all (first 5): {missing_series_metadata[:5]}. "
          f"These will get an all-masked-out (~neutral) prediction, not a crash.")

# Test series with raw DICOM files present on disk (built directly from test_series/,
# not from any pre-baked NPZ volume dataset).
def _series_has_dicoms(study_uid, series_uid, dicom_root=TEST_DICOM_DIR):
    series_dir = os.path.join(dicom_root, str(study_uid), str(series_uid))
    if not os.path.isdir(series_dir):
        return False
    with os.scandir(series_dir) as entries:
        return any(e.is_file() and e.name.lower().endswith(".dcm") for e in entries)

series_to_dicomdir = {}
for series_uid, study_uid in series_to_study.items():
    if _series_has_dicoms(study_uid, series_uid):
        series_to_dicomdir[series_uid] = os.path.join(TEST_DICOM_DIR, str(study_uid), str(series_uid))
print(f"Test series with raw DICOM files found on disk: {len(series_to_dicomdir)}")

missing_dicom = [s for s in series_to_study if s not in series_to_dicomdir]
print(f"Series missing raw DICOM files: {len(missing_dicom)}")

series_per_study = pd.Series({st: len(v) for st, v in study_to_series.items()})
print("Series-per-study stats:")
print(series_per_study.describe())

if test_series_df is not None:
    plane_col = None
    for c in ["Plane", "SeriesDescription", "orientation", "Orientation"]:
        if c in test_series_df.columns:
            plane_col = c
            break
    if plane_col:
        print(f"\nPlane-related column found: '{plane_col}' — value counts:")
        print(test_series_df[plane_col].value_counts(dropna=False).head(20))
    else:
        print("\nNo plane/orientation-like column found in test_series.csv.")


test: shape=(3, 1), columns=['StudyInstanceUID']
test_series: shape=(15, 5), columns=['StudyInstanceUID', 'SeriesInstanceUID', 'Fluid_Sensitive', 'Fat_Suppression', 'Anatomical_Plane']
sample_submission: shape=(3, 13), columns=['StudyInstanceUID', 'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
Test studies to predict (from sample_submission.csv when available): 3
Test series:  15
Test series with raw DICOM files found on disk: 15
Series missing raw DICOM files: 0
Series-per-study stats:
count    3.0
mean     5.0
std      0.0
min      5.0
25%      5.0
50%      5.0
75%      5.0
max      5.0
dtype: float64

No plane/orientation-like column found in test_series.csv.


## 4. Laterality extraction (test set)
Same DICOM-header strategy as the training notebook's laterality audit: read a single
representative DICOM per series (`stop_before_pixels=True`), pull `Laterality` /
`ImageLaterality`, aggregate to study level, and leave any study with no reliable
L/R signal as `'unknown'` (never guessed).

In [4]:
import pydicom
from concurrent.futures import ThreadPoolExecutor, as_completed

LATERALITY_WORKERS = min(16, (os.cpu_count() or 4) * 2)
TEST_LATERALITY_CACHE = WORK_DIR / "test_laterality_cache.csv"

def find_one_dicom(series_dir):
    try:
        with os.scandir(series_dir) as entries:
            for entry in entries:
                if entry.is_file() and entry.name.lower().endswith(".dcm"):
                    return entry.path
    except Exception:
        pass
    return None

def read_series_laterality(args):
    study_uid, series_uid, series_dir = args
    try:
        dicom_path = find_one_dicom(series_dir)
        if dicom_path is None:
            return {"StudyInstanceUID": study_uid, "SeriesInstanceUID": series_uid, "Laterality": "unknown"}
        ds = pydicom.dcmread(dicom_path, stop_before_pixels=True, force=True)
        value = getattr(ds, "Laterality", None) or getattr(ds, "ImageLaterality", None)
        if value is None:
            side = "unknown"
        else:
            value = str(value).strip().upper()
            side = "L" if value.startswith("L") else ("R" if value.startswith("R") else "unknown")
        return {"StudyInstanceUID": study_uid, "SeriesInstanceUID": series_uid, "Laterality": side}
    except Exception as e:
        return {"StudyInstanceUID": study_uid, "SeriesInstanceUID": series_uid, "Laterality": "unknown", "error": repr(e)}

def aggregate_study_laterality(group):
    sides = set(x for x in group["Laterality"].astype(str) if x in {"L", "R"})
    if sides == {"L"}:
        return "L"
    if sides == {"R"}:
        return "R"
    return "unknown"  # empty or conflicting -> never guessed

laterality_map = {}
if os.path.isdir(TEST_DICOM_DIR):
    jobs = []
    for row in test_series_df[["StudyInstanceUID", "SeriesInstanceUID"]].drop_duplicates().itertuples(index=False):
        series_dir = os.path.join(TEST_DICOM_DIR, str(row.StudyInstanceUID), str(row.SeriesInstanceUID))
        jobs.append((row.StudyInstanceUID, row.SeriesInstanceUID, series_dir))

    results = []
    with ThreadPoolExecutor(max_workers=LATERALITY_WORKERS) as executor:
        futures = {executor.submit(read_series_laterality, job): job for job in jobs}
        for future in as_completed(futures):
            results.append(future.result())

    laterality_series_df = pd.DataFrame(results).drop_duplicates(subset=["SeriesInstanceUID"], keep="last")
    laterality_series_df.to_csv(TEST_LATERALITY_CACHE, index=False)

    laterality_map_df = (
        laterality_series_df.groupby("StudyInstanceUID", as_index=False)
        .apply(lambda g: pd.Series({"Laterality": aggregate_study_laterality(g)}), include_groups=False)
        .reset_index(drop=True)
    )
    laterality_map = dict(zip(laterality_map_df["StudyInstanceUID"], laterality_map_df["Laterality"]))

    n_left = sum(v == "L" for v in laterality_map.values())
    n_right = sum(v == "R" for v in laterality_map.values())
    n_unknown = len(test_study_uids) - n_left - n_right
    print(f"Laterality: L={n_left}, R={n_right}, unknown={n_unknown} (of {len(test_study_uids)} test studies)")
else:
    print(f"WARNING: {TEST_DICOM_DIR} not found — all test studies will be treated as laterality 'unknown' "
          f"(canonicalization skipped for every study, never guessed).")

def get_laterality(study_uid):
    return laterality_map.get(study_uid, "unknown")


Laterality: L=1, R=0, unknown=2 (of 3 test studies)


## 5. Preprocessing utilities (verbatim from the training notebook)
`normalize_intensity`, `canonicalize_laterality`, `load_volume`, `build_25d_indices`,
`make_25d_image` — unchanged.

In [5]:
import pydicom  # required by load_volume's raw-DICOM reading below

def normalize_intensity(volume, mode="baseline", eps=1e-6):
    volume = volume.astype(np.float32)
    if mode == "baseline":
        return volume / 255.0 if volume.max() > 1.5 else volume
    elif mode == "minmax":
        lo, hi = volume.min(), volume.max()
        return (volume - lo) / (hi - lo + eps)
    elif mode == "percentile":
        lo, hi = np.percentile(volume, [1, 99])
        clipped = np.clip(volume, lo, hi)
        return (clipped - lo) / (hi - lo + eps)
    else:
        raise ValueError(f"Unknown INTENSITY_NORMALIZATION mode: {mode}")


def canonicalize_laterality(volume, side):
    """volume: (D, H, W). side: 'L' | 'R' | 'unknown'. Unknown passes through unchanged."""
    if side == "unknown":
        return volume, False
    if side == "R":
        return volume[:, :, ::-1].copy(), True
    return volume, False


def _list_series_dicom_paths(series_dir):
    paths = []
    try:
        with os.scandir(series_dir) as entries:
            for entry in entries:
                if entry.is_file() and entry.name.lower().endswith(".dcm"):
                    paths.append(entry.path)
    except Exception:
        pass
    return paths


def _dicom_slice_sort_key(ds):
    """Sort by physical position along the slice normal (ImagePositionPatient projected
    onto the cross product of the row/column direction cosines); falls back to
    InstanceNumber, then 0, if the geometry tags are missing."""
    try:
        iop = [float(v) for v in ds.ImageOrientationPatient]
        ipp = [float(v) for v in ds.ImagePositionPatient]
        row = np.array(iop[:3], dtype=np.float64)
        col = np.array(iop[3:], dtype=np.float64)
        normal = np.cross(row, col)
        return float(np.dot(np.array(ipp, dtype=np.float64), normal))
    except Exception:
        try:
            return float(ds.InstanceNumber)
        except Exception:
            return 0.0


def _frame_to_uint8(frame2d, slope, intercept, size):
    """Single (H, W) grayscale frame -> rescaled -> 0.5/99.5 percentile windowed ->
    uint8 [0, 255] -> resized to (size, size)."""
    arr = frame2d.astype(np.float32) * slope + intercept
    lo, hi = np.percentile(arr, [0.5, 99.5])
    if hi <= lo:
        lo, hi = float(arr.min()), float(arr.max())
    arr = np.clip(arr, lo, hi)
    denom = (hi - lo) if (hi - lo) > 1e-6 else 1.0
    arr = (arr - lo) / denom
    arr = np.clip(arr * 255.0, 0, 255).astype(np.uint8)

    if arr.shape != (size, size):
        try:
            import cv2
            arr = cv2.resize(arr, (size, size), interpolation=cv2.INTER_AREA)
        except Exception:
            from PIL import Image
            arr = np.array(Image.fromarray(arr).resize((size, size), Image.BILINEAR))
    return arr


def _dicom_to_uint8_slices(ds, size=VOLUME_RESIZE_HW):
    """Raw pixel_array -> one or more (size, size) uint8 grayscale slices. Tolerates
    shapes the public test set may not have shown: single-frame grayscale (H, W),
    single-frame RGB (H, W, 3/4, collapsed to grayscale), multi-frame grayscale
    (F, H, W), and multi-frame RGB (F, H, W, 3/4). Raises only if pixel_array itself
    can't be decoded (e.g. missing compressed-transfer-syntax codec) -- callers catch
    that per-file so one bad slice never kills the whole series/run."""
    raw = np.asarray(ds.pixel_array)
    slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
    intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
    n_frames = int(getattr(ds, "NumberOfFrames", 1) or 1)

    if raw.ndim == 2:
        frames2d = [raw]
    elif raw.ndim == 3:
        if raw.shape[-1] in (3, 4) and n_frames <= 1:
            frames2d = [raw[..., :3].astype(np.float32).mean(axis=-1)]  # RGB -> grayscale
        else:
            frames2d = [raw[f] for f in range(raw.shape[0])]  # (F, H, W)
    elif raw.ndim == 4:
        frames2d = [raw[f, ..., :3].astype(np.float32).mean(axis=-1) for f in range(raw.shape[0])]  # (F, H, W, 3/4)
    else:
        raise ValueError(f"Unsupported pixel_array shape {raw.shape}")

    return [_frame_to_uint8(f, slope, intercept, size) for f in frames2d]


def load_volume(series_uid, size=VOLUME_RESIZE_HW, verbose=True):
    """Builds a (D, size, size) uint8 volume directly from the raw test DICOM series
    on disk -- e.g. TEST_DICOM_DIR/<StudyInstanceUID>/<SeriesInstanceUID>/*.dcm --
    instead of reading a pre-baked "processed 3D volume" NPZ file. Any single file
    that fails to read or decode is skipped (not fatal) -- the volume is built from
    whatever slices decode successfully, so one bad DICOM in the hidden test set
    can't take down the whole submission."""
    series_dir = series_to_dicomdir.get(series_uid)
    if series_dir is None:
        return None

    paths = _list_series_dicom_paths(series_dir)
    if not paths:
        return None

    datasets = []
    for p in paths:
        try:
            datasets.append(pydicom.dcmread(p, force=True))
        except Exception as e:
            if verbose:
                print(f"  [load_volume] skipping unreadable file {p}: {type(e).__name__}: {e}")
            continue
    if not datasets:
        return None

    datasets.sort(key=_dicom_slice_sort_key)

    slices = []
    for ds in datasets:
        try:
            slices.extend(_dicom_to_uint8_slices(ds, size=size))
        except Exception as e:
            if verbose:
                sop = getattr(ds, "SOPInstanceUID", "?")
                print(f"  [load_volume] skipping undecodable slice (series={series_uid}, sop={sop}): "
                      f"{type(e).__name__}: {e}")
            continue
    if not slices:
        return None

    return np.stack(slices, axis=0)  # (D, size, size) uint8


def build_25d_indices(n_slices):
    """[i-1, i, i+1] with edge clamping. Preserves slice ordering; never shuffled."""
    triplets = []
    for i in range(n_slices):
        lo = max(i - 1, 0)
        hi = min(i + 1, n_slices - 1)
        triplets.append((lo, i, hi))
    return triplets


def make_25d_image(volume_norm, i_lo, i_mid, i_hi):
    """volume_norm: (D, H, W) float32 in [0,1]. Returns (H, W, 3) float32."""
    return np.stack([volume_norm[i_lo], volume_norm[i_mid], volume_norm[i_hi]], axis=-1)

_test = build_25d_indices(5)
assert _test[0] == (0, 0, 1) and _test[1] == (0, 1, 2) and _test[-1] == (3, 4, 4)
print("2.5D boundary-handling self-check passed.")


2.5D boundary-handling self-check passed.


## 5b. Metadata utility (verbatim from training)
Defined once, here, before anything that needs it (Sections 7’s downstream cache
primitive, and every dataset class below) — same body as the training notebook’s
Section 11 dataset-construction cell.

In [6]:
def infer_plane_from_series_row(row):
    # Inspect actual series-description-like columns rather than assuming a fixed schema.
    for col in ["Plane", "SeriesDescription", "orientation", "Orientation"]:
        if col in row and not pd.isna(row[col]):
            v = str(row[col]).lower()
            if "sag" in v: return "Sagittal"
            if "cor" in v: return "Coronal"
            if "ax" in v:  return "Axial"
    return "Sagittal"  # CONFIGURED ASSUMPTION fallback when plane cannot be determined


def infer_metadata_from_series_row(row, n_slices, max_slices):
    fluid_sensitive = 0.0
    fat_suppression = 0.0
    for col in ["FluidSensitive", "fluid_sensitive", "T2", "STIR"]:
        if col in row and not pd.isna(row[col]):
            fluid_sensitive = float(bool(row[col])); break
    for col in ["FatSuppression", "fat_suppression", "FS"]:
        if col in row and not pd.isna(row[col]):
            fat_suppression = float(bool(row[col])); break
    slice_fraction = min(n_slices / max(max_slices, 1), 1.0)
    return np.array([fluid_sensitive, fat_suppression, slice_fraction], dtype=np.float32)

print("infer_plane_from_series_row / infer_metadata_from_series_row defined.")


infer_plane_from_series_row / infer_metadata_from_series_row defined.


## 6. Download trained weights from Hugging Face

In [7]:
from pathlib import Path

# ============================================================
# Local Kaggle Dataset — NO INTERNET REQUIRED
# ============================================================

CKPT_DIR = Path("/kaggle/input/datasets/michealehab4/dinov2-knee-ckpts")

# Check that the dataset exists
if not CKPT_DIR.exists():
    raise FileNotFoundError(
        f"Kaggle checkpoint dataset not found: {CKPT_DIR}"
    )

# ------------------------------------------------------------
# Checkpoint paths
# ------------------------------------------------------------

BACKBONE_CKPT_PATH = (
    CKPT_DIR / "dinov2_336_lora_finetuned_backbone.pt"
)

FOLD_CKPT_PATHS = [
    CKPT_DIR / f"dino336_finetuned_fold_{i}_best.pt"
    for i in range(5)
]

# ------------------------------------------------------------
# Verify all files exist
# ------------------------------------------------------------

required_files = [
    BACKBONE_CKPT_PATH,
    *FOLD_CKPT_PATHS,
]

print("Checking local Kaggle checkpoints...\n")

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(
            f"❌ Missing checkpoint:\n{path}"
        )

    size_mb = path.stat().st_size / (1024 ** 2)

    print(f"✅ {path.name}")
    print(f"   Path: {path}")
    print(f"   Size: {size_mb:.2f} MB")

# ------------------------------------------------------------
# Final paths
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("Checkpoint configuration")
print("=" * 70)

print("\nBackbone checkpoint:")
print(BACKBONE_CKPT_PATH)

print("\nModel G fold checkpoints:")
for p in FOLD_CKPT_PATHS:
    print(p)

print("\n✅ All checkpoints loaded from Kaggle Dataset.")
print("🌐 Internet is NOT required.")

Checking local Kaggle checkpoints...

✅ dinov2_336_lora_finetuned_backbone.pt
   Path: /kaggle/input/datasets/michealehab4/dinov2-knee-ckpts/dinov2_336_lora_finetuned_backbone.pt
   Size: 332.66 MB
✅ dino336_finetuned_fold_0_best.pt
   Path: /kaggle/input/datasets/michealehab4/dinov2-knee-ckpts/dino336_finetuned_fold_0_best.pt
   Size: 4.03 MB
✅ dino336_finetuned_fold_1_best.pt
   Path: /kaggle/input/datasets/michealehab4/dinov2-knee-ckpts/dino336_finetuned_fold_1_best.pt
   Size: 4.03 MB
✅ dino336_finetuned_fold_2_best.pt
   Path: /kaggle/input/datasets/michealehab4/dinov2-knee-ckpts/dino336_finetuned_fold_2_best.pt
   Size: 4.03 MB
✅ dino336_finetuned_fold_3_best.pt
   Path: /kaggle/input/datasets/michealehab4/dinov2-knee-ckpts/dino336_finetuned_fold_3_best.pt
   Size: 4.03 MB
✅ dino336_finetuned_fold_4_best.pt
   Path: /kaggle/input/datasets/michealehab4/dinov2-knee-ckpts/dino336_finetuned_fold_4_best.pt
   Size: 4.03 MB

Checkpoint configuration

Backbone checkpoint:
/kaggle/input/

## 7. Build DINOv2-336 + LoRA and load the fine-tuned backbone
The saved checkpoint is the state dict of the backbone **after** LoRA was applied
(`apply_lora_to_vit`) and fine-tuned — so the architecture must be rebuilt with LoRA
applied first, using the exact same rank/alpha/dropout/target keywords, before the
checkpoint's keys (e.g. `...query.base.weight`, `...query.lora_A`) will match.

In [8]:
from transformers import AutoImageProcessor, AutoModel

def load_dinov2(local_path=DINOV2_LOCAL_PATH, resolution=224):
    if not os.path.isdir(local_path):
        raise FileNotFoundError(
            f"Local DINOv2 checkpoint not found at {local_path}. "
            f"This notebook is configured to use the local checkpoint only."
        )
    processor = AutoImageProcessor.from_pretrained(local_path)
    model = AutoModel.from_pretrained(local_path)
    if hasattr(processor, "size"):
        if isinstance(processor.size, dict):
            processor.size = {"height": resolution, "width": resolution}
        else:
            processor.size = resolution
    model.eval()
    return processor, model


def load_dinov2_336(local_path=DINOV2_LOCAL_PATH):
    processor, model = load_dinov2(local_path, resolution=336)
    model.config.image_size = 336
    return processor, model


class LoRALinear(nn.Module):
    """y = W0 x + (alpha / r) * B(A(dropout(x))). W0 stays frozen."""
    def __init__(self, base_linear, rank=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base = base_linear
        for p in self.base.parameters():
            p.requires_grad_(False)
        in_f, out_f = base_linear.in_features, base_linear.out_features
        self.rank = rank
        self.scaling = alpha / rank
        self.lora_A = nn.Parameter(torch.zeros(rank, in_f))
        self.lora_B = nn.Parameter(torch.zeros(out_f, rank))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        base_out = self.base(x)
        lora_update = self.dropout(x) @ self.lora_A.t() @ self.lora_B.t()
        return base_out + self.scaling * lora_update


def apply_lora_to_vit(model, rank=8, alpha=16, dropout=0.05, target_keywords=("query", "key", "value")):
    replaced = 0
    for _, module in list(model.named_modules()):
        for attr_name, child in list(module.named_children()):
            if isinstance(child, nn.Linear) and any(k in attr_name.lower() for k in target_keywords):
                setattr(module, attr_name, LoRALinear(child, rank=rank, alpha=alpha, dropout=dropout))
                replaced += 1
    print(f"LoRA applied to {replaced} attention projection layers "
          f"(rank={rank}, alpha={alpha}, targets={list(target_keywords)}).")
    if replaced == 0:
        print("WARNING: no matching Linear layers found for LoRA — checkpoint keys will not match.")
    return model


DINOV2_AVAILABLE = os.path.isdir(DINOV2_LOCAL_PATH)
print(f"Local DINOv2 checkpoint present at {DINOV2_LOCAL_PATH} -> {DINOV2_AVAILABLE}")
assert DINOV2_AVAILABLE, "This notebook requires the local DINOv2 base architecture/model files (same as training)."

dino_processor_336, dino_model_336 = load_dinov2_336(DINOV2_LOCAL_PATH)
dino_model_336 = apply_lora_to_vit(
    dino_model_336, rank=LORA_RANK, alpha=LORA_ALPHA, dropout=LORA_DROPOUT,
    target_keywords=LORA_TARGET_KEYWORDS,
)
# LayerNorms were also fine-tuned (affine params) -- architecture-wise this changes
# nothing (LayerNorm modules already exist), it only matters for which keys had
# requires_grad=True during training; the checkpoint contains their fine-tuned values
# regardless, so a plain state_dict load picks them up correctly.

state_dict = torch.load(BACKBONE_CKPT_PATH, map_location="cpu")
load_result = dino_model_336.load_state_dict(state_dict, strict=True)
print("Backbone state dict loaded (strict=True): no missing/unexpected keys.")

dino_model_336.to(DEVICE)
for p in dino_model_336.parameters():
    p.requires_grad_(False)
dino_model_336.eval()

n_trainable = sum(p.numel() for p in dino_model_336.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in dino_model_336.parameters())
embed_dim = dino_model_336.config.hidden_size if hasattr(dino_model_336.config, "hidden_size") else 768

print(f"DINOv2 resolution: {DINO_RESOLUTION}")
print(f"embedding dimension: {embed_dim}")
print("LoRA: loaded")
print(f"trainable parameters: {n_trainable}")
assert n_trainable == 0, "Backbone must be fully frozen for inference."
assert embed_dim == 768, "Expected 768-D DINOv2 embeddings."


The image processor of type `BitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Local DINOv2 checkpoint present at /kaggle/input/models/metaresearch/dinov2/pytorch/base/1 -> True


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

LoRA applied to 48 attention projection layers (rank=8, alpha=16, targets=['query', 'key', 'value', 'dense']).
Backbone state dict loaded (strict=True): no missing/unexpected keys.
DINOv2 resolution: 336
embedding dimension: 768
LoRA: loaded
trainable parameters: 0


## 8. DINOv2 slice-embedding extraction primitive (live, no disk cache)
Identical to the training notebook's Stage-A extraction function, but nothing
downstream persists its output to disk -- every call re-reads the raw DICOM series
and re-runs the backbone. This is what `DicomStudyDataset` (Section 11) calls per
series on every study it serves, so submission inference is always DICOM -> DINOv2,
never a cached .npz.

In [9]:
@torch.inference_mode()
def extract_embeddings_for_series(series_uid, processor, model, resolution=DINO_RESOLUTION,
                                    intensity_mode=INTENSITY_NORMALIZATION, laterality_side="unknown",
                                    apply_laterality=False, batch_size=None):
    batch_size = batch_size or DINO_BATCH_SIZE
    volume = load_volume(series_uid)
    if volume is None:
        return None
    if apply_laterality:
        volume, _ = canonicalize_laterality(volume, laterality_side)
    vol_norm = normalize_intensity(volume, mode=intensity_mode)
    n_slices = vol_norm.shape[0]
    triplets = build_25d_indices(n_slices)

    cls_embeds = []
    for start in range(0, n_slices, batch_size):
        batch_triplets = triplets[start:start + batch_size]
        imgs = [make_25d_image(vol_norm, lo, mid, hi) for (lo, mid, hi) in batch_triplets]
        imgs_uint8 = [(im * 255).astype(np.uint8) for im in imgs]
        inputs = processor(images=imgs_uint8, return_tensors="pt", size={"height": resolution, "width": resolution})
        pixel_values = inputs["pixel_values"].to(next(model.parameters()).device)
        autocast_device = "cuda" if pixel_values.is_cuda else "cpu"
        with torch.autocast(device_type=autocast_device, enabled=pixel_values.is_cuda):
            try:
                out = model(pixel_values=pixel_values, interpolate_pos_encoding=(resolution != 224))
            except TypeError:
                out = model(pixel_values=pixel_values)
        cls = out.last_hidden_state[:, 0, :]
        cls_embeds.append(cls.float().cpu().numpy())

    cls_embeds = np.concatenate(cls_embeds, axis=0)  # [N_slices, 768]
    positions = np.arange(n_slices)
    return {"cls": cls_embeds, "positions": positions}

print("Stage-A extraction primitive defined (live DICOM -> DINOv2, no disk cache).")


Stage-A extraction primitive defined (live DICOM -> DINOv2, no disk cache).


## 9. Exact Model G architecture and checkpoint loading

In [10]:
class ModelG(nn.Module):
    """Exact reproduction of the trained label-specific multi-view fusion model."""
    def __init__(self, cfg=MODEL_G_CONFIG, metadata_dim=METADATA_DIM, n_labels=N_LABELS):
        super().__init__()
        embed_dim = cfg["embed_dim"]
        hidden_dim = cfg["hidden_dim"]
        view_dim = cfg["view_dim"]
        dropout = cfg["dropout"]
        cls_hidden = cfg["classifier_hidden_dim"]

        self.input_projection = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
        )
        self.metadata_projection = nn.Sequential(
            nn.Linear(metadata_dim, view_dim),
            nn.LayerNorm(view_dim),
        )
        self.series_projection = nn.Sequential(
            nn.Linear(hidden_dim + view_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
        )
        self.to_view = nn.Linear(hidden_dim, view_dim)

        self.label_queries = nn.Parameter(torch.randn(n_labels, view_dim) * 0.02)
        self.attention_temperature = nn.Parameter(torch.ones(n_labels))

        self.label_heads = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(view_dim),
                nn.Linear(view_dim, cls_hidden),
                nn.GELU(),
                nn.Dropout(dropout / 2.0),
                nn.Linear(cls_hidden, 1),
            )
            for _ in range(n_labels)
        ])
        self.dropout = nn.Dropout(dropout)
        self.n_labels = n_labels

    def forward(self, slice_embeddings, slice_mask, metadata, series_mask):
        B, S, T, E = slice_embeddings.shape
        x = self.input_projection(slice_embeddings)
        x = self.dropout(x)

        mask = slice_mask.unsqueeze(-1).float()
        summed = (x * mask).sum(dim=2)
        counts = mask.sum(dim=2).clamp(min=1.0)
        series_repr = summed / counts

        meta = self.metadata_projection(metadata)
        fused = torch.cat([series_repr, meta], dim=-1)
        fused = self.series_projection(fused)
        view_repr = self.to_view(fused)

        logits_out, attn_out = [], []
        for label_idx in range(self.n_labels):
            query = self.label_queries[label_idx]
            temperature = self.attention_temperature[label_idx].abs() + 1e-6
            scores = (view_repr * query).sum(dim=-1) / temperature
            scores = scores.masked_fill(series_mask <= 0, -1e4)
            weights = F.softmax(scores, dim=1)
            fused_label = (weights.unsqueeze(-1) * view_repr).sum(dim=1)
            logit = self.label_heads[label_idx](fused_label).squeeze(-1)
            logits_out.append(logit)
            attn_out.append(weights)

        logits = torch.stack(logits_out, dim=1)   # [B,12]
        attn = torch.stack(attn_out, dim=1)       # [B,12,S]
        return logits, attn


fold_models = []
for fold_idx, ckpt_path in enumerate(FOLD_CKPT_PATHS):
    model = ModelG().to(DEVICE)
    state_dict = torch.load(ckpt_path, map_location=DEVICE)
    load_result = model.load_state_dict(state_dict, strict=True)
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    fold_models.append(model)
    print(f"Fold {fold_idx} loaded ✓  (no missing/unexpected keys)")

with torch.no_grad():
    _dummy = torch.zeros(1, MAX_SERIES_PER_STUDY, MAX_SLICES_PER_SERIES, MODEL_G_CONFIG["embed_dim"], device=DEVICE)
    _dummy_mask = torch.zeros(1, MAX_SERIES_PER_STUDY, MAX_SLICES_PER_SERIES, device=DEVICE)
    _dummy_meta = torch.zeros(1, MAX_SERIES_PER_STUDY, METADATA_DIM, device=DEVICE)
    _dummy_smask = torch.ones(1, MAX_SERIES_PER_STUDY, device=DEVICE)
    _logits, _ = fold_models[0](_dummy, _dummy_mask, _dummy_meta, _dummy_smask)
    print(f"Model G output shape check: {tuple(_logits.shape)} (expected (1, {N_LABELS}))")
    assert _logits.shape == (1, N_LABELS)

print(f"\nAll {len(fold_models)} Model G folds loaded.")


Fold 0 loaded ✓  (no missing/unexpected keys)
Fold 1 loaded ✓  (no missing/unexpected keys)
Fold 2 loaded ✓  (no missing/unexpected keys)
Fold 3 loaded ✓  (no missing/unexpected keys)
Fold 4 loaded ✓  (no missing/unexpected keys)
Model G output shape check: (1, 12) (expected (1, 12))

All 5 Model G folds loaded.


## 10. Checkpoint / configuration summary

In [11]:
print("=" * 60)
print("INFERENCE MODEL")
print("=" * 60)
print()
print("Backbone:")
print("    DINOv2-336")
print("    LoRA fine-tuned")
print(f"    checkpoint: {HF_BACKBONE_FILE.split('/')[-1]}")
print("    frozen: YES")
print("    embedding dim: 768")
print()
print("Model G:")
print(f"    folds: {N_FOLDS}")
print("    fold checkpoints: dino336_finetuned_fold_*_best.pt")
print("    ensemble: probability mean")
print()
print("Preprocessing:")
print("    2.5D: YES")
print(f"    resolution: {DINO_RESOLUTION}")
print(f"    intensity: {INTENSITY_NORMALIZATION}")
print(f"    laterality canonicalization: {USE_LATERALITY_CANONICALIZATION}")
print("    reports: NO")
print("    LLM labels: NO")
print("=" * 60)


INFERENCE MODEL

Backbone:
    DINOv2-336
    LoRA fine-tuned
    checkpoint: dinov2_336_lora_finetuned_backbone.pt
    frozen: YES
    embedding dim: 768

Model G:
    folds: 5
    fold checkpoints: dino336_finetuned_fold_*_best.pt
    ensemble: probability mean

Preprocessing:
    2.5D: YES
    resolution: 336
    intensity: baseline
    laterality canonicalization: True
    reports: NO
    LLM labels: NO


## 10b. Sanity check: reproduce the gold-set AUC with THESE downloaded weights
Public score came back at 0.499 (≈ random). Before touching test-set plumbing, this
verifies the model/preprocessing pipeline itself is correct: it re-extracts DINOv2
embeddings **from scratch, using only what this notebook downloaded/built above** — no
training-time cache is read — for the ~58 gold-labeled `train.csv` studies, runs the
same 5-fold Model G ensemble, and computes macro AUC with the exact `compute_per_label_auc`
/ `macro_auc` functions the training notebook used.

- If this reproduces ≈ 0.7966 → the model/backbone/checkpoints/preprocessing are all
  correct, and a 0.499 public score points at test-time data plumbing (wrong
  `TEST_DICOM_DIR`, series↔study ID mismatches, or the submission's
  column/row order) rather than the model.
- If this comes back near 0.5 too → the bug is in this notebook's model pipeline
  itself (wrong checkpoint, backbone/LoRA mismatch, preprocessing divergence), and the
  cells below will help localize where.

In [12]:
from sklearn.metrics import roc_auc_score

TRAIN_CSV_SANITY        = f"{COMPETITION_DIR}/train.csv"
TRAIN_SERIES_CSV_SANITY = f"{COMPETITION_DIR}/train_series.csv"
TRAIN_MRI_DIR_SANITY    = "/kaggle/input/datasets/barun2104/rsna-knee-mri-processed-3d-volumes"
TRAIN_DICOM_DIR_SANITY  = os.path.join(COMPETITION_DIR, "train_series")
GOLD_SANITY_CACHE = INFERENCE_CACHE_ROOT / "gold_sanity_check"
GOLD_SANITY_CACHE.mkdir(parents=True, exist_ok=True)

train_df_sanity        = safe_read_csv(TRAIN_CSV_SANITY)
train_series_df_sanity = safe_read_csv(TRAIN_SERIES_CSV_SANITY)
assert train_df_sanity is not None and train_series_df_sanity is not None, \
    "train.csv / train_series.csv are required for the gold-set sanity check."

missing_label_cols = [l for l in LABELS if l not in train_df_sanity.columns]
assert not missing_label_cols, f"train.csv missing expected label columns: {missing_label_cols}"

# Gold studies = complete label rows only (same definition as build_gold_targets in training).
gold_targets = {}
for _, row in train_df_sanity.iterrows():
    study = row.get("StudyInstanceUID")
    if study is None or pd.isna(study):
        continue
    vals = row[LABELS].values.astype(float)
    if np.isnan(vals).any():
        continue
    gold_targets[study] = vals

gold_studies = sorted(gold_targets.keys())
print(f"Gold-labeled train studies with complete labels: {len(gold_studies)} (spec expects ~58)")
assert len(gold_studies) > 0, "No gold-labeled studies found -- cannot run the sanity check."

train_series_to_study = dict(zip(train_series_df_sanity["SeriesInstanceUID"],
                                   train_series_df_sanity["StudyInstanceUID"]))
train_study_to_series = {}
for s, st in train_series_to_study.items():
    train_study_to_series.setdefault(st, []).append(s)

train_npz_files = sorted(glob.glob(os.path.join(TRAIN_MRI_DIR_SANITY, "*.npz"))) \
    if os.path.isdir(TRAIN_MRI_DIR_SANITY) else []
train_series_to_npz = {p.split("/")[-1].removesuffix(".npz"): p for p in train_npz_files}
print(f"Train NPZ volumes found on disk: {len(train_npz_files)}")

gold_series_needed = sorted({s for st in gold_studies for s in train_study_to_series.get(st, [])})
missing_gold_npz = [s for s in gold_series_needed if s not in train_series_to_npz]
print(f"Gold-study series needed: {len(gold_series_needed)}, missing NPZ: {len(missing_gold_npz)}")


Gold-labeled train studies with complete labels: 58 (spec expects ~58)
Train NPZ volumes found on disk: 24371
Gold-study series needed: 336, missing NPZ: 0


In [13]:
# Laterality for the gold train studies (separate map from the TEST one in Section 4;
# same DICOM-header strategy, applied here to train_series.csv / train_series DICOM tree).
gold_laterality_map = {}
if os.path.isdir(TRAIN_DICOM_DIR_SANITY):
    jobs = []
    needed_study_series = train_series_df_sanity[
        train_series_df_sanity["StudyInstanceUID"].isin(gold_studies)
    ][["StudyInstanceUID", "SeriesInstanceUID"]].drop_duplicates()
    for row in needed_study_series.itertuples(index=False):
        series_dir = os.path.join(TRAIN_DICOM_DIR_SANITY, str(row.StudyInstanceUID), str(row.SeriesInstanceUID))
        jobs.append((row.StudyInstanceUID, row.SeriesInstanceUID, series_dir))

    results = []
    with ThreadPoolExecutor(max_workers=LATERALITY_WORKERS) as executor:
        futures = {executor.submit(read_series_laterality, job): job for job in jobs}
        for future in as_completed(futures):
            results.append(future.result())

    gold_laterality_series_df = pd.DataFrame(results).drop_duplicates(subset=["SeriesInstanceUID"], keep="last")
    gold_laterality_map_df = (
        gold_laterality_series_df.groupby("StudyInstanceUID", as_index=False)
        .apply(lambda g: pd.Series({"Laterality": aggregate_study_laterality(g)}), include_groups=False)
        .reset_index(drop=True)
    )
    gold_laterality_map = dict(zip(gold_laterality_map_df["StudyInstanceUID"], gold_laterality_map_df["Laterality"]))
    n_left = sum(v == "L" for v in gold_laterality_map.values())
    n_right = sum(v == "R" for v in gold_laterality_map.values())
    print(f"Gold-study laterality: L={n_left}, R={n_right}, "
          f"unknown={len(gold_studies) - n_left - n_right} (of {len(gold_studies)})")
else:
    print(f"WARNING: {TRAIN_DICOM_DIR_SANITY} not found -- gold studies treated as laterality 'unknown'.")


def get_gold_laterality(study_uid):
    return gold_laterality_map.get(study_uid, "unknown")


@torch.inference_mode()
def extract_embeddings_train(series_uid):
    path = train_series_to_npz.get(series_uid)
    if path is None:
        return None
    with np.load(path) as npz:
        volume = npz["data"]
    side = get_gold_laterality(train_series_to_study.get(series_uid))
    if USE_LATERALITY_CANONICALIZATION:
        volume, _ = canonicalize_laterality(volume, side)
    vol_norm = normalize_intensity(volume, mode=INTENSITY_NORMALIZATION)
    n_slices = vol_norm.shape[0]
    triplets = build_25d_indices(n_slices)

    cls_embeds = []
    for start in range(0, n_slices, DINO_BATCH_SIZE):
        batch_triplets = triplets[start:start + DINO_BATCH_SIZE]
        imgs = [make_25d_image(vol_norm, lo, mid, hi) for (lo, mid, hi) in batch_triplets]
        imgs_uint8 = [(im * 255).astype(np.uint8) for im in imgs]
        inputs = dino_processor_336(images=imgs_uint8, return_tensors="pt",
                                     size={"height": DINO_RESOLUTION, "width": DINO_RESOLUTION})
        pixel_values = inputs["pixel_values"].to(DEVICE)
        with torch.autocast(device_type="cuda", enabled=pixel_values.is_cuda):
            try:
                out = dino_model_336(pixel_values=pixel_values, interpolate_pos_encoding=(DINO_RESOLUTION != 224))
            except TypeError:
                out = dino_model_336(pixel_values=pixel_values)
        cls_embeds.append(out.last_hidden_state[:, 0, :].float().cpu().numpy())

    return {"embeddings": np.concatenate(cls_embeds, axis=0), "positions": np.arange(n_slices)}


t0 = time.time()
n_extracted, n_skipped = 0, 0
for series_uid in gold_series_needed:
    cp = GOLD_SANITY_CACHE / f"{series_uid}.npz"
    if cp.exists():
        continue
    result = extract_embeddings_train(series_uid)
    if result is None:
        n_skipped += 1
        continue
    np.savez_compressed(cp, embeddings=result["embeddings"], positions=result["positions"])
    n_extracted += 1
print(f"Fresh gold-set extraction: {n_extracted} series extracted, {n_skipped} skipped (no NPZ), "
      f"in {time.time() - t0:.1f}s")


Gold-study laterality: L=11, R=16, unknown=31 (of 58)
Fresh gold-set extraction: 336 series extracted, 0 skipped (no NPZ), in 53.5s


In [14]:
class GoldStudyDataset(Dataset):
    """Same shape/logic as DicomStudyDataset (Section 11) but for the gold TRAIN
    studies, reading ONLY the freshly-built GOLD_SANITY_CACHE, and attaching the
    known gold target vector for AUC scoring."""
    def __init__(self, study_uids, cache_dir=GOLD_SANITY_CACHE, representation="cls",
                 max_series=MAX_SERIES_PER_STUDY, max_slices=MAX_SLICES_PER_SERIES):
        self.study_uids = study_uids
        self.cache_dir = cache_dir
        self.max_series = max_series
        self.max_slices = max_slices

    def __len__(self):
        return len(self.study_uids)

    def __getitem__(self, idx):
        study = self.study_uids[idx]
        series_list = train_study_to_series.get(study, [])[: self.max_series]

        S, T, E = self.max_series, self.max_slices, MODEL_G_CONFIG["embed_dim"]
        slice_embeddings = np.zeros((S, T, E), dtype=np.float32)
        slice_mask = np.zeros((S, T), dtype=np.float32)
        metadata = np.zeros((S, METADATA_DIM), dtype=np.float32)
        series_mask = np.zeros((S,), dtype=np.float32)

        for s_i, series_uid in enumerate(series_list):
            cp = self.cache_dir / f"{series_uid}.npz"
            if not cp.exists():
                continue
            with np.load(cp) as npz:
                emb = npz["embeddings"][: self.max_slices]
            n = emb.shape[0]
            slice_embeddings[s_i, :n] = emb
            slice_mask[s_i, :n] = 1.0
            series_mask[s_i] = 1.0
            row = {}
            match = train_series_df_sanity[train_series_df_sanity["SeriesInstanceUID"] == series_uid]
            if len(match):
                row = match.iloc[0]
            metadata[s_i] = infer_metadata_from_series_row(row, n, MAX_SLICES_FOR_METADATA_NORM)

        return {
            "study": study,
            "slice_embeddings": torch.from_numpy(slice_embeddings),
            "slice_mask": torch.from_numpy(slice_mask),
            "metadata": torch.from_numpy(metadata),
            "series_mask": torch.from_numpy(series_mask),
            "target": torch.from_numpy(gold_targets[study].astype(np.float32)),
        }


def collate_gold(batch):
    out = {}
    for k in ["slice_embeddings", "slice_mask", "metadata", "series_mask", "target"]:
        out[k] = torch.stack([b[k] for b in batch], dim=0)
    out["study"] = [b["study"] for b in batch]
    return out


def compute_per_label_auc(y_true, y_pred, labels=LABELS):
    results = {}
    for i, label in enumerate(labels):
        yt, yp = y_true[:, i], y_pred[:, i]
        if len(np.unique(yt)) < 2:
            results[label] = np.nan
        else:
            results[label] = roc_auc_score(yt, yp)
    return results


def macro_auc(per_label_auc_dict):
    vals = [v for v in per_label_auc_dict.values() if not np.isnan(v)]
    return float(np.mean(vals)) if vals else np.nan


gold_ds = GoldStudyDataset(gold_studies)
gold_loader = DataLoader(gold_ds, batch_size=8, shuffle=False, collate_fn=collate_gold)

fold_pred_matrices = []
y_true_ref, studies_ref = None, None
with torch.inference_mode():
    for fold_idx, model in enumerate(fold_models):
        studies_this, y_true_this, y_pred_this = [], [], []
        for batch in gold_loader:
            se = batch["slice_embeddings"].to(DEVICE)
            sm = batch["slice_mask"].to(DEVICE)
            md_ = batch["metadata"].to(DEVICE)
            smask = batch["series_mask"].to(DEVICE)
            logits, _ = model(se, sm, md_, smask)
            probs = torch.sigmoid(logits).float().cpu().numpy()
            studies_this.extend(batch["study"])
            y_true_this.append(batch["target"].numpy())
            y_pred_this.append(probs)
        y_true_this = np.concatenate(y_true_this)
        y_pred_this = np.concatenate(y_pred_this)
        fold_pred_matrices.append(y_pred_this)
        studies_ref, y_true_ref = studies_this, y_true_this

        fold_auc = macro_auc(compute_per_label_auc(y_true_this, y_pred_this))
        print(f"Fold {fold_idx} macro AUC on gold set: {fold_auc:.4f}")

mean_pred_gold = np.mean(fold_pred_matrices, axis=0)
gold_per_label = compute_per_label_auc(y_true_ref, mean_pred_gold)
gold_macro = macro_auc(gold_per_label)

print()
print("=" * 60)
print("GOLD-SET SANITY CHECK (5-fold mean ensemble, fresh extraction)")
print("=" * 60)
print(f"Macro AUC: {gold_macro:.4f}   (reference from training: 0.7966)")
for label, auc in gold_per_label.items():
    print(f"  {label:20s} {auc if np.isnan(auc) else round(auc, 4)}")
print("=" * 60)

if np.isnan(gold_macro):
    print("\nCould not compute an AUC at all -- check that gold NPZ volumes / DICOMs were found above.")
elif gold_macro > 0.70:
    print("\n=> Reproduces the training-time score. The model pipeline is correct.")
    print("   A bad public LB score points at TEST-time data (TEST_DICOM_DIR paths,")
    print("   series<->study mapping, or the submission's column/row order) -- not the model.")
elif gold_macro < 0.60:
    print("\n=> Does NOT reproduce the training-time score -- the bug is in THIS pipeline")
    print("   (checkpoint mismatch, backbone/LoRA loading, preprocessing divergence). Check the")
    print("   'trainable parameters: 0' / 'no missing/unexpected keys' prints in Sections 7 and 9 above,")
    print("   and confirm HF_REPO_ID is serving the files you expect (not a stale/partial upload).")
else:
    print("\n=> Partial reproduction -- inspect the per-label breakdown above for which labels degraded.")


Fold 0 macro AUC on gold set: 0.7910
Fold 1 macro AUC on gold set: 0.8215
Fold 2 macro AUC on gold set: 0.7294
Fold 3 macro AUC on gold set: 0.7958
Fold 4 macro AUC on gold set: 0.7435

GOLD-SET SANITY CHECK (5-fold mean ensemble, fresh extraction)
Macro AUC: 0.7966   (reference from training: 0.7966)
  ACL                  0.7904
  MCL                  0.6485
  Medial Meniscus      0.7873
  Lateral Meniscus     0.7391
  Medial OA            0.8822
  Lateral OA           0.7524
  PF OA                0.7915
  Effusion             0.8894
  Synovitis            0.871
  Baker's              0.808
  Contusion            0.7733
  Fracture             0.8264

=> Reproduces the training-time score. The model pipeline is correct.
   A bad public LB score points at TEST-time data (TEST_DICOM_DIR paths,
   series<->study mapping, or the submission's column/row order) -- not the model.


## 10c. Direct, uncached probe on the 3 real `sample_submission.csv` studies
No cache read/write anywhere in this cell — every embedding is computed live, in memory,
straight from the downloaded DINOv2-336 LoRA backbone, and fed straight into the 5
Model G folds. This exercises the exact real TEST-set series↔study lookup
(`series_to_study` / `study_to_series` / `series_to_dicomdir` built in Sections 3–4) end to
end for the 3 actual studies in `sample_submission.csv`, so any ID-mapping or
missing-NPZ problem on the real test set shows up directly here.

In [15]:
PROBE_STUDIES = (
    sample_sub_df["StudyInstanceUID"].tolist()
    if sample_sub_df is not None
    else [
        "1.2.826.0.1.3680043.8.498.10047035057544427318018579121635276191",
        "1.2.826.0.1.3680043.8.498.10062861783145312629332250977456991776",
        "1.2.826.0.1.3680043.8.498.10067514707072572280263481548497591402",
    ]
)[:3]
print(f"Probing {len(PROBE_STUDIES)} studies:")
for s in PROBE_STUDIES:
    print(" ", s)


@torch.inference_mode()
def build_study_tensor_live(study_uid, max_series=MAX_SERIES_PER_STUDY, max_slices=MAX_SLICES_PER_SERIES):
    """Extracts DINOv2 embeddings for every series of this study directly (no disk
    cache read or write) and assembles the exact [S,T,E]/[S,T]/[S,METADATA_DIM]/[S]
    tensors Model G expects. Returns None-embedding series as all-zero + masked-out,
    exactly like DicomStudyDataset does for a missing/unavailable series."""
    series_list = study_to_series.get(study_uid, [])[:max_series]
    S, T, E = max_series, max_slices, MODEL_G_CONFIG["embed_dim"]
    slice_embeddings = np.zeros((S, T, E), dtype=np.float32)
    slice_mask = np.zeros((S, T), dtype=np.float32)
    metadata = np.zeros((S, METADATA_DIM), dtype=np.float32)
    series_mask = np.zeros((S,), dtype=np.float32)

    per_series_info = []
    for s_i, series_uid in enumerate(series_list):
        has_dicom = series_uid in series_to_dicomdir
        side = get_laterality(study_uid)
        try:
            result = extract_embeddings_for_series(
                series_uid, dino_processor_336, dino_model_336, resolution=DINO_RESOLUTION,
                intensity_mode=INTENSITY_NORMALIZATION, laterality_side=side,
                apply_laterality=USE_LATERALITY_CANONICALIZATION,
            ) if has_dicom else None
        except Exception as e:
            # This is a diagnostic probe -- it must never be able to crash the whole
            # notebook (and therefore block the real submission cells below it) just
            # because one hidden-set series has an unreadable/unusual DICOM.
            print(f"  [probe] series {series_uid} failed, leaving masked-out: {type(e).__name__}: {e}")
            result = None

        if result is None:
            per_series_info.append({"series": series_uid, "has_dicom": has_dicom, "n_slices": 0, "embed_shape": None})
            continue

        try:
            emb = result["cls"][:max_slices]
            n = emb.shape[0]
            slice_embeddings[s_i, :n] = emb
            slice_mask[s_i, :n] = 1.0
            series_mask[s_i] = 1.0
            row = {}
            match = test_series_df[test_series_df["SeriesInstanceUID"] == series_uid] if test_series_df is not None else None
            if match is not None and len(match):
                row = match.iloc[0]
            metadata[s_i] = infer_metadata_from_series_row(row, n, MAX_SLICES_FOR_METADATA_NORM)
            per_series_info.append({"series": series_uid, "has_dicom": has_dicom, "n_slices": n, "embed_shape": emb.shape})
        except Exception as e:
            print(f"  [probe] series {series_uid} metadata/assign failed, leaving masked-out: {type(e).__name__}: {e}")
            slice_embeddings[s_i] = 0.0
            slice_mask[s_i] = 0.0
            series_mask[s_i] = 0.0
            metadata[s_i] = 0.0
            per_series_info.append({"series": series_uid, "has_dicom": has_dicom, "n_slices": 0, "embed_shape": None})

    tensors = {
        "slice_embeddings": torch.from_numpy(slice_embeddings).unsqueeze(0).to(DEVICE),
        "slice_mask": torch.from_numpy(slice_mask).unsqueeze(0).to(DEVICE),
        "metadata": torch.from_numpy(metadata).unsqueeze(0).to(DEVICE),
        "series_mask": torch.from_numpy(series_mask).unsqueeze(0).to(DEVICE),
    }
    return tensors, series_list, per_series_info


probe_rows = []
for study_uid in PROBE_STUDIES:
    try:
        print()
        print("=" * 60)
        print(f"Study: {study_uid}")
        tensors, series_list, per_series_info = build_study_tensor_live(study_uid)
        print(f"  series found for this study: {len(series_list)}")
        for info in per_series_info:
            print(f"    series {info['series']}: has_dicom={info['has_dicom']}, "
                  f"n_slices={info['n_slices']}, embedding_shape={info['embed_shape']}")

        if len(series_list) == 0:
            print("  WARNING: no series mapped to this study in test_series.csv -- "
                  "check StudyInstanceUID <-> SeriesInstanceUID join.")
        elif all(info["embed_shape"] is None for info in per_series_info):
            print("  WARNING: every series mapped to this study is missing its raw DICOM files on disk -- "
                  "check TEST_DICOM_DIR.")

        fold_probs = []
        for fold_idx, model in enumerate(fold_models):
            logits, _ = model(tensors["slice_embeddings"], tensors["slice_mask"],
                               tensors["metadata"], tensors["series_mask"])
            probs = torch.sigmoid(logits).float().cpu().numpy()[0]
            fold_probs.append(probs)
            print(f"  fold {fold_idx} probs: {np.round(probs, 3).tolist()}")

        mean_probs = np.mean(fold_probs, axis=0)
        print(f"  MEAN ENSEMBLE probs: {np.round(mean_probs, 3).tolist()}")
        print(f"  min={mean_probs.min():.4f}  max={mean_probs.max():.4f}  "
              f"NaN={np.isnan(mean_probs).any()}  Inf={np.isinf(mean_probs).any()}")
        print(f"  distance from all-0.5 baseline (mean |p-0.5|): {np.mean(np.abs(mean_probs - 0.5)):.4f}")

        probe_rows.append({"StudyInstanceUID": study_uid, **{l: mean_probs[i] for i, l in enumerate(LABELS)}})
    except Exception as e:
        # This whole section is a diagnostic sanity check -- it must never be able to
        # abort the notebook (and thus block the real submission cells below it).
        print(f"  [probe] study {study_uid} failed entirely, skipping: {type(e).__name__}: {e}")
        continue

print()
probe_df = pd.DataFrame(probe_rows)
print(probe_df)

if (probe_df[LABELS].sub(0.5).abs() < 0.01).all(axis=None):
    print("\nWARNING: every probability is ~0.5 for all 3 studies -- Model G is producing "
          "essentially uninformative output for real test data even though embeddings were "
          "extracted. Compare this against Section 10b's gold-set result: if 10b scored well "
          "but this is flat 0.5, the discrepancy is almost certainly in what's being fed into "
          "Model G here (metadata columns not matching test_series.csv's real schema, a plane/"
          "view mismatch, or masked-out series) rather than in DINOv2 or the checkpoints.")


Probing 3 studies:
  1.2.826.0.1.3680043.8.498.10047035057544427318018579121635276191
  1.2.826.0.1.3680043.8.498.10062861783145312629332250977456991776
  1.2.826.0.1.3680043.8.498.10067514707072572280263481548497591402

Study: 1.2.826.0.1.3680043.8.498.10047035057544427318018579121635276191
  series found for this study: 5
    series 1.2.826.0.1.3680043.8.498.11580656442259111255675562605155903947: has_dicom=True, n_slices=34, embedding_shape=(34, 768)
    series 1.2.826.0.1.3680043.8.498.17811502614030631664517622518906646132: has_dicom=True, n_slices=30, embedding_shape=(30, 768)
    series 1.2.826.0.1.3680043.8.498.30565395595045942404081022062489758495: has_dicom=True, n_slices=30, embedding_shape=(30, 768)
    series 1.2.826.0.1.3680043.8.498.32856494541816845805947850304653662081: has_dicom=True, n_slices=30, embedding_shape=(30, 768)
    series 1.2.826.0.1.3680043.8.498.44334485654554877495800965096767429689: has_dicom=True, n_slices=48, embedding_shape=(48, 768)
  fold 0 probs

## 11. Test dataset construction (live DICOM, no embedding cache)

In [16]:
class DicomStudyDataset(Dataset):
    """Computes DINOv2 embeddings directly from the raw test DICOM series for every
    __getitem__ call. No embedding cache (.npz) is read or written anywhere in this
    class -- every study's embeddings come straight from that study's .dcm files on
    THIS run, so there's no way a stale/pre-existing cache silently gets reused."""
    def __init__(self, study_uids, processor, model, representation="cls",
                 max_series=MAX_SERIES_PER_STUDY, max_slices=MAX_SLICES_PER_SERIES):
        self.study_uids = study_uids
        self.processor = processor
        self.model = model
        self.representation = representation
        self.max_series = max_series
        self.max_slices = max_slices

    def __len__(self):
        return len(self.study_uids)

    @torch.inference_mode()
    def __getitem__(self, idx):
        study = self.study_uids[idx]
        series_list = study_to_series.get(study, [])[: self.max_series]

        S, T, E = self.max_series, self.max_slices, MODEL_G_CONFIG["embed_dim"]
        slice_embeddings = np.zeros((S, T, E), dtype=np.float32)
        slice_mask = np.zeros((S, T), dtype=np.float32)
        metadata = np.zeros((S, METADATA_DIM), dtype=np.float32)
        series_mask = np.zeros((S,), dtype=np.float32)

        for s_i, series_uid in enumerate(series_list):
            if series_uid not in series_to_dicomdir:
                continue  # no raw DICOM files on disk for this series -- leave masked out

            side = get_laterality(series_to_study.get(series_uid))
            try:
                result = extract_embeddings_for_series(
                    series_uid, self.processor, self.model, resolution=DINO_RESOLUTION,
                    intensity_mode=INTENSITY_NORMALIZATION, laterality_side=side,
                    apply_laterality=USE_LATERALITY_CANONICALIZATION,
                )
            except Exception as e:
                # One bad series (unreadable/corrupt DICOMs, unexpected shape, etc. in
                # the hidden test set) must never take down the whole submission -- log
                # it and leave this series masked-out, same as a missing series.
                print(f"  [DicomStudyDataset] study={study} series={series_uid} failed, "
                      f"leaving masked-out: {type(e).__name__}: {e}")
                result = None
            if result is None:
                continue
            try:
                emb = result["cls"][: self.max_slices]
                if emb.shape[-1] != E:
                    continue
                n = emb.shape[0]
                slice_embeddings[s_i, :n] = emb
                slice_mask[s_i, :n] = 1.0
                series_mask[s_i] = 1.0
                row = {}
                if test_series_df is not None:
                    match = test_series_df[test_series_df["SeriesInstanceUID"] == series_uid]
                    if len(match):
                        row = match.iloc[0]
                metadata[s_i] = infer_metadata_from_series_row(row, n, MAX_SLICES_FOR_METADATA_NORM)
            except Exception as e:
                # Embeddings extracted fine but something odd happened in metadata lookup
                # or slot assignment -- undo this series' slot rather than crash the run.
                print(f"  [DicomStudyDataset] study={study} series={series_uid} metadata/assign "
                      f"failed, leaving masked-out: {type(e).__name__}: {e}")
                slice_embeddings[s_i] = 0.0
                slice_mask[s_i] = 0.0
                series_mask[s_i] = 0.0
                metadata[s_i] = 0.0

        return {
            "study": study,
            "slice_embeddings": torch.from_numpy(slice_embeddings),
            "slice_mask": torch.from_numpy(slice_mask),
            "metadata": torch.from_numpy(metadata),
            "series_mask": torch.from_numpy(series_mask),
        }


def collate_studies(batch):
    out = {}
    for k in ["slice_embeddings", "slice_mask", "metadata", "series_mask"]:
        out[k] = torch.stack([b[k] for b in batch], dim=0)
    out["study"] = [b["study"] for b in batch]
    return out

print("DicomStudyDataset / collate_studies defined (live DICOM -> DINOv2, no embedding cache).")


DicomStudyDataset / collate_studies defined (live DICOM -> DINOv2, no embedding cache).


## 12. Inference pipeline: run Model G directly against live DICOM embeddings
`infer_studies` builds a `DicomStudyDataset` (which computes each series' DINOv2
embeddings from its raw .dcm files on the fly -- no cache file is read or written)
and runs the 5-fold mean-probability ensemble over those studies.

In [17]:
@torch.inference_mode()
def infer_studies(study_uids, batch_size=8):
    ds = DicomStudyDataset(study_uids, dino_processor_336, dino_model_336, representation=REPRESENTATION)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, collate_fn=collate_studies)

    all_studies, fold_probs = [], [[] for _ in fold_models]
    for batch in loader:
        se = batch["slice_embeddings"].to(DEVICE)
        sm = batch["slice_mask"].to(DEVICE)
        md_ = batch["metadata"].to(DEVICE)
        smask = batch["series_mask"].to(DEVICE)
        all_studies.extend(batch["study"])
        for fold_idx, model in enumerate(fold_models):
            logits, _ = model(se, sm, md_, smask)
            probs = torch.sigmoid(logits).float().cpu().numpy()
            fold_probs[fold_idx].append(probs)

    fold_probs = [np.concatenate(fp, axis=0) for fp in fold_probs]  # each [N, 12]
    mean_probs = np.mean(fold_probs, axis=0)
    return all_studies, mean_probs, fold_probs

print("Inference pipeline functions defined (Model G reads live DICOM-derived embeddings only).")


Inference pipeline functions defined (Model G reads live DICOM-derived embeddings only).


## 13. Smoke test
Runs a small number of studies through the full pipeline (cache build → 5 folds →
mean ensemble) and checks the output shape and value ranges before the full run.

In [18]:
if SMOKE_TEST:
    smoke_studies = test_study_uids[:SMOKE_N_STUDIES]
    print(f"Smoke test on {len(smoke_studies)} studies: {smoke_studies}")

    for study in smoke_studies:
        series_list = study_to_series.get(study, [])
        n_with_dicom = sum(1 for s in series_list[:MAX_SERIES_PER_STUDY] if s in series_to_dicomdir)
        print(f"  study {study}: {len(series_list)} series, {n_with_dicom} with raw DICOM files on disk")

    studies_out, mean_probs, fold_probs = infer_studies(smoke_studies)
    print("\nModel G output shape (per fold logits->probs):", fold_probs[0].shape)
    print("Final mean-probability shape:", mean_probs.shape, f"(expected ({len(smoke_studies)}, {N_LABELS}))")

    for i, study in enumerate(studies_out):
        p = mean_probs[i]
        print(f"  {study}: shape={p.shape}, min={p.min():.4f}, max={p.max():.4f}, "
              f"NaN={np.isnan(p).any()}, Inf={np.isinf(p).any()}")

    assert mean_probs.shape == (len(smoke_studies), N_LABELS)
    assert not np.isnan(mean_probs).any(), "NaNs found in smoke-test probabilities."
    assert not np.isinf(mean_probs).any(), "Infs found in smoke-test probabilities."
    assert (mean_probs >= 0).all() and (mean_probs <= 1).all(), "Probabilities out of [0, 1] range."
    print("\nSmoke test PASSED.")
else:
    print("SMOKE_TEST is False -- skipping.")


Smoke test on 2 studies: ['1.2.826.0.1.3680043.8.498.10047035057544427318018579121635276191', '1.2.826.0.1.3680043.8.498.10062861783145312629332250977456991776']
  study 1.2.826.0.1.3680043.8.498.10047035057544427318018579121635276191: 5 series, 5 with raw DICOM files on disk
  study 1.2.826.0.1.3680043.8.498.10062861783145312629332250977456991776: 5 series, 5 with raw DICOM files on disk

Model G output shape (per fold logits->probs): (2, 12)
Final mean-probability shape: (2, 12) (expected (2, 12))
  1.2.826.0.1.3680043.8.498.10047035057544427318018579121635276191: shape=(12,), min=0.1965, max=0.8180, NaN=False, Inf=False
  1.2.826.0.1.3680043.8.498.10062861783145312629332250977456991776: shape=(12,), min=0.1872, max=0.9076, NaN=False, Inf=False

Smoke test PASSED.


## 14. Full inference

In [19]:
SMOKE_TEST = False  # smoke test already passed above; run the complete test set now

print(f"Running full inference on {len(test_study_uids)} test studies (live DICOM -> DINOv2, no embedding cache)...")

n_missing_dicom = sum(
    1 for study in test_study_uids
    for series_uid in study_to_series.get(study, [])[:MAX_SERIES_PER_STUDY]
    if series_uid not in series_to_dicomdir
)
if n_missing_dicom > 0:
    print(f"WARNING: {n_missing_dicom} series had no raw DICOM files on disk and were skipped "
          f"(their slots remain zero-padded/masked-out in Model G's input -- no features were fabricated).")

t0 = time.time()
all_studies, mean_probs, fold_probs = infer_studies(test_study_uids)
print(f"Inference finished in {time.time() - t0:.1f}s for {len(all_studies)} studies.")

# Warn + repair rather than hard-assert here too -- the submission-building cell
# (Section 15) does its own final repair pass anyway, but catching it here first
# gives an earlier, clearer signal in the logs about which studies were affected.
bad_mask = ~np.isfinite(mean_probs)
n_bad = int(bad_mask.sum())
if n_bad > 0:
    bad_studies = sorted({all_studies[i] for i in np.where(bad_mask.any(axis=1))[0]})
    print(f"WARNING: {n_bad} non-finite probability values across {len(bad_studies)} studies "
          f"(first 5: {bad_studies[:5]}) -- replacing with 0.5 (max-uncertainty).")
    mean_probs = np.where(bad_mask, 0.5, mean_probs)

n_out_of_range = int(((mean_probs < 0) | (mean_probs > 1)).sum())
if n_out_of_range > 0:
    print(f"WARNING: {n_out_of_range} probability values outside [0, 1] -- clipping.")
    mean_probs = np.clip(mean_probs, 0.0, 1.0)

if n_bad == 0 and n_out_of_range == 0:
    print("Full inference sanity checks passed (no NaN/Inf, all probabilities in [0, 1]).")


Running full inference on 3 test studies (live DICOM -> DINOv2, no embedding cache)...
Inference finished in 10.3s for 3 studies.
Full inference sanity checks passed (no NaN/Inf, all probabilities in [0, 1]).


## 15. Build and validate the submission CSV
Column names, label ordering, and row format follow `sample_submission.csv` exactly
when it's available; otherwise falls back to `StudyInstanceUID` + the 12 `LABELS`
columns in the order defined above.

In [20]:
pred_df = pd.DataFrame({"StudyInstanceUID": all_studies})
pred_df["StudyInstanceUID"] = pred_df["StudyInstanceUID"].astype(str).str.strip()
for i, label in enumerate(LABELS):
    pred_df[label] = mean_probs[:, i]

if sample_sub_df is not None:
    sub_columns = list(sample_sub_df.columns)
    assert "StudyInstanceUID" in sub_columns, "sample_submission.csv must contain StudyInstanceUID."
    missing_cols = [c for c in sub_columns if c not in pred_df.columns and c != "StudyInstanceUID"]
    if missing_cols:
        print(f"WARNING: sample_submission.csv has columns not produced by this model: {missing_cols}. "
              f"Inspect the competition's expected schema -- these are left as NaN.")
    sample_sub_key = sample_sub_df[["StudyInstanceUID"]].copy()
    sample_sub_key["StudyInstanceUID"] = sample_sub_key["StudyInstanceUID"].astype(str).str.strip()
    submission = sample_sub_key.merge(pred_df, on="StudyInstanceUID", how="left")
    submission = submission[sub_columns] if not missing_cols else submission
else:
    print("sample_submission.csv not available -- using StudyInstanceUID + LABELS as the submission schema.")
    submission = pred_df

print("submission.shape:", submission.shape)
print(submission.head())
print(submission.dtypes)
print("Missing values:\n", submission.isna().sum())

label_cols = [c for c in submission.columns if c != "StudyInstanceUID"]

# --- Self-healing safety net (never let a coverage/numeric edge case throw and kill
# the whole scoring run -- warn loudly and repair instead, same philosophy as the
# reference notebook's explicit "0.5 max-uncertainty fallback for studies with no
# usable series") ---
n_dupes = int(submission["StudyInstanceUID"].duplicated().sum())
if n_dupes > 0:
    print(f"WARNING: {n_dupes} duplicate StudyInstanceUID rows in submission -- keeping first occurrence.")
    submission = submission.drop_duplicates(subset=["StudyInstanceUID"], keep="first").reset_index(drop=True)

n_missing_rows = 0
if sample_sub_df is not None:
    n_missing_rows = len(sample_sub_df) - len(submission)
    if n_missing_rows != 0:
        print(f"WARNING: submission has {len(submission)} rows, sample_submission.csv has "
              f"{len(sample_sub_df)}. This should not happen (every sample_submission study is "
              f"in test_study_uids) -- inspect StudyInstanceUID formatting/dtype mismatches above.")

n_missing_preds = int(submission[label_cols].isna().any(axis=1).sum())
if n_missing_preds > 0:
    print(f"WARNING: {n_missing_preds} studies got no prediction at all (StudyInstanceUID present in "
          f"sample_submission.csv but missing from the inference pass) -- filling with 0.5 "
          f"(max-uncertainty) for those rows rather than failing the whole submission.")
    submission[label_cols] = submission[label_cols].fillna(0.5)

prob_values = submission[label_cols].values.astype(float)
n_bad = int((~np.isfinite(prob_values)).sum())
if n_bad > 0:
    print(f"WARNING: {n_bad} non-finite (NaN/Inf) probability values -- replacing with 0.5.")
    prob_values = np.where(np.isfinite(prob_values), prob_values, 0.5)

n_out_of_range = int(((prob_values < 0) | (prob_values > 1)).sum())
if n_out_of_range > 0:
    print(f"WARNING: {n_out_of_range} probability values outside [0, 1] -- clipping.")
    prob_values = np.clip(prob_values, 0.0, 1.0)

submission[label_cols] = prob_values
print("Probability range (post-repair):", (float(prob_values.min()), float(prob_values.max())))

if sample_sub_df is not None and n_missing_rows == 0 and n_dupes == 0 and n_missing_preds == 0 and n_bad == 0 and n_out_of_range == 0:
    print("Submission integrity checks passed with no repairs needed.")

SUBMISSION_PATH = WORK_DIR / "submission.csv"
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"\nSaved: {SUBMISSION_PATH}")

DIAGNOSTIC_PATH = WORK_DIR / "inference_predictions.csv"
pred_df.to_csv(DIAGNOSTIC_PATH, index=False)
print(f"Saved (diagnostic): {DIAGNOSTIC_PATH}")


submission.shape: (3, 13)
                                    StudyInstanceUID       ACL       MCL  \
0  1.2.826.0.1.3680043.8.498.10047035057544427318...  0.196893  0.196506   
1  1.2.826.0.1.3680043.8.498.10062861783145312629...  0.354757  0.187223   
2  1.2.826.0.1.3680043.8.498.10067514707072572280...  0.172170  0.155478   

   Medial Meniscus  Lateral Meniscus  Medial OA  Lateral OA     PF OA  \
0         0.736346          0.324274   0.704878    0.540475  0.761301   
1         0.777832          0.491692   0.856107    0.781657  0.855725   
2         0.788591          0.367655   0.755616    0.586020  0.818911   

   Effusion  Synovitis   Baker's  Contusion  Fracture  
0  0.787885   0.818013  0.596408   0.327693  0.217722  
1  0.855774   0.907645  0.709786   0.425273  0.302202  
2  0.636672   0.775225  0.359206   0.320105  0.188630  
StudyInstanceUID     object
ACL                 float32
MCL                 float32
Medial Meniscus     float32
Lateral Meniscus    float32
Medial OA   

## 16. Final summary

In [21]:
print("=" * 60)
print("INFERENCE COMPLETE")
print("=" * 60)
print()
print(f"Studies inferred: {len(all_studies)}")
print("Model: DINOv2-336 LoRA + Model G")
print(f"Folds: {N_FOLDS}")
print("Ensemble: probability mean")
print()
print("Submission:")
print(f"    {SUBMISSION_PATH}")
print()
print("Shape:")
print(f"    {submission.shape[0]} × {submission.shape[1]}")
print()
print("NaNs:")
print(f"    {int(submission[label_cols].isna().sum().sum())}")
print()
print("Infs:")
print(f"    {int(np.isinf(prob_values).sum())}")
print()
print("Probability range:")
print(f"    [{float(np.nanmin(prob_values)):.4f}, {float(np.nanmax(prob_values)):.4f}]")
print("=" * 60)


INFERENCE COMPLETE

Studies inferred: 3
Model: DINOv2-336 LoRA + Model G
Folds: 5
Ensemble: probability mean

Submission:
    /kaggle/working/submission.csv

Shape:
    3 × 13

NaNs:
    0

Infs:
    0

Probability range:
    [0.1555, 0.9076]
